In [5]:
!pip install transformers
!pip install protobuf

DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import torch
import torchaudio
import numpy as np
import pandas as pd
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

print("✅ Imports done")

✅ Imports done


In [19]:
# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR   = "/Users/abey/Documents/ACCENT_CLASSIFICATION"   # ← change this
MODELS_DIR = os.path.join(BASE_DIR, "models")

ACCENT_REFERENCES = {
    "american": os.path.join(BASE_DIR, "reference", "american.wav"),
    "british" : os.path.join(BASE_DIR, "reference", "british.wav"),
    "indian"  : os.path.join(BASE_DIR, "reference", "indian.wav"),
}

TARGET_ACCENT = "american"
TARGET_THRESHOLD = 0.75
CHARACTER_MAP = None

print(f"Target accent         : {TARGET_ACCENT}")
print(f"Target threshold      : {TARGET_THRESHOLD}")
print(f"Accent references     : {list(ACCENT_REFERENCES.keys())}")
print("✅ Paths and thresholds set")

Target accent         : american
Target threshold      : 0.75
Accent references     : ['american', 'british', 'indian']
✅ Paths and thresholds set


In [16]:
# ============================================================
# CELL 3 — Load wav2vec2 xlsr-53
# ============================================================
MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"

print(f"Loading {MODEL_NAME}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
wav2vec2          = Wav2Vec2Model.from_pretrained(MODEL_NAME)
wav2vec2.eval()

print(f"✅ wav2vec2 loaded: {MODEL_NAME}")

Loading facebook/wav2vec2-large-xlsr-53...
✅ wav2vec2 loaded: facebook/wav2vec2-large-xlsr-53


In [17]:
# ============================================================
# CELL 4 — Embedding extraction
# ============================================================
def get_accent_embedding(audio_path):
    wav, sr = torchaudio.load(audio_path)

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != 16000:
        wav = torchaudio.transforms.Resample(sr, 16000)(wav)

    if wav.shape[-1] < 160:
        raise ValueError(f"Audio too short: {audio_path}")

    inputs = feature_extractor(
        wav.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    with torch.no_grad():
        outputs = wav2vec2(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1).squeeze()
    return embedding


def cosine_sim(emb1, emb2):
    e1 = emb1 / torch.norm(emb1)
    e2 = emb2 / torch.norm(emb2)
    return round(float(torch.dot(e1, e2)), 4)

print("✅ get_accent_embedding and cosine_sim defined")

✅ get_accent_embedding and cosine_sim defined


In [20]:
# ============================================================
# CELL 5 — Startup validation
# ============================================================
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

for accent_name, ref_path in ACCENT_REFERENCES.items():
    if not os.path.exists(ref_path):
        raise FileNotFoundError(
            f"Accent reference not found for '{accent_name}': {ref_path}\n"
            f"Provide a clean audio clip of any speaker with that accent."
        )
print(f"✅ All {len(ACCENT_REFERENCES)} accent reference files found")

if TARGET_ACCENT not in ACCENT_REFERENCES:
    raise ValueError(
        f"TARGET_ACCENT '{TARGET_ACCENT}' not in ACCENT_REFERENCES.\n"
        f"Available: {list(ACCENT_REFERENCES.keys())}"
    )

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

total = len(model_folders) * len(model_samples[model_folders[0]])
print(f"\nReady: {len(model_folders)} models × {len(model_samples[model_folders[0]])} samples = {total} evaluations")

✅ All 3 accent reference files found
✅ Models found: ['m1', 'm2']
   m1: 2 samples
   m2: 2 samples
✅ All models have identical filenames

Ready: 2 models × 2 samples = 4 evaluations


In [21]:
# ============================================================
# CELL 6 — Pre-compute all accent reference embeddings
# ============================================================
print("\nExtracting accent reference embeddings...")
accent_ref_embeddings = {}

for accent_name, ref_path in ACCENT_REFERENCES.items():
    try:
        emb = get_accent_embedding(ref_path)
        accent_ref_embeddings[accent_name] = emb
        print(f"  ✅ {accent_name}: {ref_path}")
    except Exception as e:
        print(f"  ❌ {accent_name} failed: {e}")
        accent_ref_embeddings[accent_name] = None

print("✅ Reference embeddings ready")


Extracting accent reference embeddings...


/Users/abey/miniconda3/envs/utmos/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


  ✅ american: /Users/abey/Documents/ACCENT_CLASSIFICATION/reference/american.wav
  ✅ british: /Users/abey/Documents/ACCENT_CLASSIFICATION/reference/british.wav
  ✅ indian: /Users/abey/Documents/ACCENT_CLASSIFICATION/reference/indian.wav
✅ Reference embeddings ready


In [23]:
# ============================================================
# CELL 7 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    embeddings = {}
    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        audio_path  = os.path.join(MODELS_DIR, model, wav_file)

        try:
            emb = get_accent_embedding(audio_path)
            embeddings[sample_name] = emb
            print(f"  ✅ Embedded: {sample_name}")
        except Exception as e:
            print(f"  ❌ Embedding failed: {sample_name} — {e}")
            embeddings[sample_name] = None

    if CHARACTER_MAP is not None:
        char_groups = CHARACTER_MAP
    else:
        char_groups = {"all": [os.path.splitext(f)[0] for f in model_samples[model]]}

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        emb         = embeddings.get(sample_name)

        if emb is None:
            row = {
                "Model"         : model,
                "Sample"        : sample_name,
                "Character"     : "—",
                "Closest Accent": "—",
                "Target Pass"   : "⚠️ ERROR",
                "Final Pass"    : "⚠️ ERROR",
            }
            for accent_name in ACCENT_REFERENCES:
                row[f"Proximity_{accent_name}"] = None
            results.append(row)
            continue

        proximities = {}
        for accent_name, ref_emb in accent_ref_embeddings.items():
            if ref_emb is not None:
                proximities[accent_name] = cosine_sim(emb, ref_emb)
            else:
                proximities[accent_name] = None

        target_proximity = proximities.get(TARGET_ACCENT)
        target_pass      = (target_proximity >= TARGET_THRESHOLD) if target_proximity is not None else None

        valid_prox     = {k: v for k, v in proximities.items() if v is not None}
        closest_accent = max(valid_prox, key=valid_prox.get) if valid_prox else "—"

        character = "unknown"
        for char_name, char_samples in char_groups.items():
            if sample_name in char_samples:
                character = char_name
                break

        if target_pass is False:
            final_pass = f"❌ FAIL (Accent drift from {TARGET_ACCENT})"
        elif target_pass is None:
            final_pass = "⚠️ ERROR"
        else:
            final_pass = "✅ PASS"

        prox_str = " | ".join([f"{k}: {v}" for k, v in proximities.items() if v is not None])
        print(f"  {sample_name} | {prox_str} | Closest: {closest_accent} → {final_pass}")

        row = {
            "Model"         : model,
            "Sample"        : sample_name,
            "Character"     : character,
            "Closest Accent": closest_accent,
            "Target Pass"   : "✅" if target_pass else "❌" if target_pass is not None else "—",
            "Final Pass"    : final_pass,
        }
        for accent_name in ACCENT_REFERENCES:
            row[f"Proximity_{accent_name}"] = proximities.get(accent_name)

        results.append(row)

print("\n\nAll evaluations complete.")


Model: m1
  ✅ Embedded: clean_baseline 2
  ✅ Embedded: clean_baseline
  clean_baseline 2 | american: 1.0 | british: 1.0 | indian: 1.0 | Closest: american → ✅ PASS
  clean_baseline | american: 1.0 | british: 1.0 | indian: 1.0 | Closest: american → ✅ PASS

Model: m2
  ✅ Embedded: clean_baseline 2
  ✅ Embedded: clean_baseline
  clean_baseline 2 | american: 1.0 | british: 1.0 | indian: 1.0 | Closest: american → ✅ PASS
  clean_baseline | american: 1.0 | british: 1.0 | indian: 1.0 | Closest: american → ✅ PASS


All evaluations complete.


In [25]:
# ============================================================
# CELL 8 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)

proximity_cols = [f"Proximity_{a}" for a in ACCENT_REFERENCES]

print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample", "Character", "Closest Accent",
    *proximity_cols,
    "Final Pass"
]].to_string(index=False))

print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df     = df[df["Model"] == model]
    total        = len(model_df)

    on_target    = (model_df["Closest Accent"] == TARGET_ACCENT).sum()
    label_consist = f"{on_target}/{total}"
    target_fails  = (model_df["Final Pass"].str.contains("FAIL")).sum()

    failing_df   = model_df[model_df["Final Pass"].str.contains("FAIL")]
    drift_counts = failing_df["Closest Accent"].value_counts().to_dict()
    drift_str    = ", ".join([f"{k}: {v}" for k, v in drift_counts.items()]) if drift_counts else "—"

    row = {
        "Model"             : model,
        "Segments"          : total,
        "Label Consistency" : label_consist,
        "Target Drift Fails": target_fails,
        "Drift Breakdown"   : drift_str,
    }

    for accent_name in ACCENT_REFERENCES:
        col  = f"Proximity_{accent_name}"
        vals = model_df[col].dropna()
        row[f"Median_{accent_name}"] = round(vals.median(), 4) if len(vals) > 0 else None

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print("\n========== MODEL RANKING ==========")
print(f"Primary   → Label Consistency (segments where Closest Accent == {TARGET_ACCENT})")
print(f"Tiebreak1 → Median Proximity_{TARGET_ACCENT} (highest first)\n")

def parse_rate(rate_str):
    if rate_str == "—":
        return -1
    return int(rate_str.split("/")[0])

summary_df["_label_consist_num"] = summary_df["Label Consistency"].apply(parse_rate)
summary_df["_median_target"]     = summary_df[f"Median_{TARGET_ACCENT}"].fillna(-999)

ranking = summary_df.sort_values(
    by=["_label_consist_num", "_median_target"],
    ascending=[False, False]
)[[
    "Model", "Label Consistency", f"Median_{TARGET_ACCENT}",
    "Target Drift Fails", "Drift Breakdown"
]]
print(ranking.to_string(index=False))

if CHARACTER_MAP is not None:
    print("\n========== CHARACTER CONSISTENCY REPORT ==========")
    for model in model_folders:
        model_df = df[df["Model"] == model]
        print(f"\nModel: {model}")
        for char in CHARACTER_MAP:
            char_df = model_df[model_df["Character"] == char]
            if len(char_df) == 0:
                continue
            on_target = (char_df["Closest Accent"] == TARGET_ACCENT).sum()
            print(f"  {char} → segments: {len(char_df)} | "
                  f"On target: {on_target}/{len(char_df)} | "
                  f"Median {TARGET_ACCENT}: {round(char_df[f'Proximity_{TARGET_ACCENT}'].median(), 4)}")

print("\n========== WHAT TO LOOK FOR ==========")
print(f"Label Consistency    → % of segments where closest accent was {TARGET_ACCENT}")
print(f"                       change TARGET_ACCENT in Cell 2 to switch target")
print(f"Proximity_{TARGET_ACCENT}  → cosine similarity to {TARGET_ACCENT} reference — below {TARGET_THRESHOLD} = drifting")
print(f"Closest Accent       → which accent bucket each segment falls into")
print(f"Drift Breakdown      → where failing segments drifted — tells you which accent is leaking")
print(f"Target Drift Fails   → segments where Proximity_{TARGET_ACCENT} < {TARGET_THRESHOLD}")
print(f"\nThresholds:")
print(f"  Target proximity  : >= {TARGET_THRESHOLD}")
print(f"  Target accent     : {TARGET_ACCENT} — change in Cell 2")


========== FULL PER-SEGMENT RESULTS ==========
Model           Sample Character Closest Accent  Proximity_american  Proximity_british  Proximity_indian Final Pass
   m1 clean_baseline 2       all       american                 1.0                1.0               1.0     ✅ PASS
   m1   clean_baseline       all       american                 1.0                1.0               1.0     ✅ PASS
   m2 clean_baseline 2       all       american                 1.0                1.0               1.0     ✅ PASS
   m2   clean_baseline       all       american                 1.0                1.0               1.0     ✅ PASS

========== MODEL COMPARISON SUMMARY ==========
Model  Segments Label Consistency  Target Drift Fails Drift Breakdown  Median_american  Median_british  Median_indian
   m1         2               2/2                   0               —              1.0             1.0            1.0
   m2         2               2/2                   0               —              1.0  